In [2]:
from typing import Any, Callable, Optional, Union

from pprint import pprint
from datetime import datetime
from pathlib import Path
import os

from torch.utils.tensorboard import SummaryWriter
import pandas as pd
import pytz
import numpy as np
import safetensors.torch as safetensors
import tqdm.notebook as tqdm
import torch
import torch.utils.data as torchdata
import torch.nn as nn
import torchmetrics
import yaml

from flatiron.core.dataset import Dataset
from flatiron.core.types import Compiled, Filepath, Getter

import torch._dynamo
torch._dynamo.config.suppress_errors = True

Filepath = Union[str, Path]


from flatiron.core.tools import get_tensorboard_project
from flatiron.torch.tools import ModelCheckpoint, get_callbacks, TorchDataset, _execute_epoch

In [3]:
def train(
    device,      # type: str
    model,       # type: torch.nn.Module
    optimizer,   # type: torch.optim.Optimizer
    loss,        # type: torch.nn.Module
    metrics,     # type: list[torch.nn.Module]
    callbacks,   # type: Callbacks
    train_data,  # type: Dataset
    test_data,   # type: Dataset
    params,      # type: dict
):
    # type: (...) -> None
    '''
    Train Torch model.

    Args:
        device (str): Device to compile to.
        model (torch.nn.Module): Model to be compiled.
        optimizer (dict): Optimizer config for compilation.
        loss (str): Loss to be compiled.
        metrics (list[str]): Metrics function to be compiled.
        callbacks (dict): Dict of callbacks.
        train_data (Dataset): Training dataset.
        test_data (Dataset): Test dataset.
        params (dict): Training params.
    '''
    checkpoint = callbacks['checkpoint']  # type: Any
    writer = callbacks['tensorboard']
    batch_size = params['batch_size']

    device = torch.device(device)
    torch.manual_seed(params['seed'])
    model = model.to(device)

    train_loader = torchdata.DataLoader(
        TorchDataset.monkey_patch(train_data), batch_size=batch_size
    )  # type: torchdata.DataLoader
    test_loader = torchdata.DataLoader(
        TorchDataset.monkey_patch(test_data), batch_size=batch_size
    )  # type: torchdata.DataLoader

    kwargs = dict(
        model=model,
        optimizer=optimizer,
        loss_func=loss.to(device),
        device=device,
        metrics_funcs=[x.to(device) for x in metrics],
        writer=writer,
    )
    for i in tqdm.trange(params['epochs']):
        _execute_epoch(
            epoch=i, mode='train', data_loader=train_loader,
            checkpoint=checkpoint, **kwargs
        )
        _execute_epoch(epoch=i, mode='test', data_loader=test_loader, **kwargs)
        if checkpoint.save_freq == 'epoch':
            checkpoint.save(model, i)

In [4]:
# DATA
data_kwargs = dict(
    directory='/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
    label_axis=-1,
    labels=['a'],
)
data = Dataset.read_directory(**data_kwargs)
train_data, test_data = data.train_test_split()
print('DATA')
pprint(data_kwargs)

DATA
{'directory': '/mnt/storage/projects/unet001/dset002/p-unet001_s-dset002_d-glom_v001',
 'label_axis': -1,
 'labels': ['a']}


In [5]:
# MODEL ARCHITECTURE
class Model(nn.Module):
    def __init__(self, input_channels, output_channels):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Conv2d(
                in_channels=input_channels, out_channels=output_channels,
                kernel_size=(3, 3), dtype=torch.float16, padding=1
            ),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.layer_stack(x)

In [6]:
# MODEL
model_kwargs = dict(
    input_channels=3,
    output_channels=1,
)
model = Model(**model_kwargs)
print('MODEL')
pprint(model_kwargs)

MODEL
{'input_channels': 3, 'output_channels': 1}


In [7]:
# CALLBACKS
tb = get_tensorboard_project(
    project='unet001',
    root='/mnt/storage/projects',
)
print('TENSORBOARD')
pprint(tb)

callback_kwargs = dict(
    log_directory=tb['log_dir'],
    checkpoint_pattern=tb['checkpoint_pattern'],
    checkpoint_params=dict(save_freq='epoch'),
)
callbacks = get_callbacks(**callback_kwargs)
print()
print('CALLBACKS')
pprint(callback_kwargs)

# TRAIN KWARGS
train_kwargs = dict(
    device='cuda',
    model=torch.compile(model),
    optimizer=torch.optim.SGD(
        model.parameters(),
        lr=0.001,
    ),
    loss=nn.modules.loss.MSELoss(),
    metrics=[
        torchmetrics.MeanMetric(),
    ],
    callbacks=callbacks,
    train_data=train_data,
    test_data=test_data,
    params=dict(
        epochs=10,
        seed=42,
        batch_size=32,
    )
)
print()
print('TRAIN')
pprint(train_kwargs)

# TRAIN
print()
train(**train_kwargs)

TENSORBOARD
{'checkpoint_pattern': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-18-22-05/models/p-unet001_d-2025-02-26_t-18-22-05_e-{epoch:03d}.keras',
 'log_dir': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-18-22-05',
 'model_dir': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-18-22-05/models',
 'root_dir': '/mnt/storage/projects/unet001/tensorboard'}

CALLBACKS
{'checkpoint_params': {'save_freq': 'epoch'},
 'checkpoint_pattern': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-18-22-05/models/p-unet001_d-2025-02-26_t-18-22-05_e-{epoch:03d}.keras',
 'log_directory': '/mnt/storage/projects/unet001/tensorboard/d-2025-02-26_t-18-22-05'}

TRAIN
{'callbacks': {'checkpoint': <flatiron.torch.tools.ModelCheckpoint object at 0x7fbe581e7730>,
               'tensorboard': <torch.utils.tensorboard.writer.SummaryWriter object at 0x7fbce89a2020>},
 'device': 'cuda',
 'loss': MSELoss(),
 'metrics': [MeanMetric()],
 'model': OptimizedModule(
  (_orig_m

  0%|          | 0/10 [00:00<?, ?it/s]

W0226 18:22:09.171000 705038 torch/_inductor/utils.py:1137] [0/0] Not enough SMs to use max_autotune_gemm mode
W0226 18:22:09.238000 705038 torch/_dynamo/convert_frame.py:1233] WON'T CONVERT forward /tmp/ipykernel_705038/2959099092.py line 13 
W0226 18:22:09.238000 705038 torch/_dynamo/convert_frame.py:1233] due to: 
W0226 18:22:09.238000 705038 torch/_dynamo/convert_frame.py:1233] Traceback (most recent call last):
W0226 18:22:09.238000 705038 torch/_dynamo/convert_frame.py:1233]   File "/home/ubuntu/pdm/envs/pdm-kVbOHlCT-dev-3.10/lib/python3.10/site-packages/torch/_dynamo/convert_frame.py", line 1164, in __call__
W0226 18:22:09.238000 705038 torch/_dynamo/convert_frame.py:1233]     result = self._inner_convert(
W0226 18:22:09.238000 705038 torch/_dynamo/convert_frame.py:1233]   File "/home/ubuntu/pdm/envs/pdm-kVbOHlCT-dev-3.10/lib/python3.10/site-packages/torch/_dynamo/convert_frame.py", line 547, in __call__
W0226 18:22:09.238000 705038 torch/_dynamo/convert_frame.py:1233]     retur

In [22]:
!exa --tree /mnt/storage/projects/unet001/tensorboard/d-2025*

/mnt/storage/projects/unet001/tensorboard/d-2025-02-25_t-13-44-28
├── events.out.tfevents.1740509068.5abe6464f7f1.258531.1
└── models
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-000.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-001.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-002.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-003.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-004.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-005.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-006.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-007.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-008.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-009.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-010.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-011.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-012.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28_e-013.safetensors
   ├── p-unet001_d-2025-02-25_t-13-44-28